In [7]:
from google.colab import drive
import pandas as pd
import numpy as np
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk

# Download necessary NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

drive.mount('/content/drive')




[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
#insert your paths to the csv files below

true = pd.read_csv("/content/drive/MyDrive/data/raw/True.csv")
fake = pd.read_csv("/content/drive/MyDrive/data/raw/Fake.csv")
true["label"] = 1
fake["label"] = 0

df = pd.concat([true, fake], ignore_index=True)

In [9]:
df = df.fillna("")





In [10]:
# 3. TEXT CLEANING FUNCTION
def clean_text(text):
    # Lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove emails
    text = re.sub(r'\S+@\S+', '', text)
    # Remove special characters and numbers (keep letters and spaces)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Remove extra whitespace
    text = ' '.join(text.split())
    return text


# 4. APPLY CLEANING
df['text_clean'] = df['text'].apply(clean_text)
df['title_clean'] = df['title'].apply(clean_text)

# 5. REMOVE STOPWORDS & LEMMATIZATION (optional but recommended)
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [11]:
def preprocess_text(text):
    tokens = text.split()
    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    # Lemmatize
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return ' '.join(tokens)

df['text_processed'] = df['text_clean'].apply(preprocess_text)
df['title_processed'] = df['title_clean'].apply(preprocess_text)

In [12]:
# 6. FEATURE ENGINEERING
df['text_length'] = df['text'].apply(lambda x: len(str(x).split()))
df['title_length'] = df['title'].apply(lambda x: len(str(x).split()))
df['has_url'] = df['text'].apply(lambda x: 1 if 'http' in str(x) else 0)

In [13]:
# 7. ENCODE CATEGORICAL (subject)
df = pd.get_dummies(df, columns=['subject'], prefix='subject')

# 8. HANDLE DUPLICATES (based on your EDA findings)
df = df.drop_duplicates(subset=['text_processed'], keep='first')


In [14]:
# 9. TRAIN-TEST SPLIT

from sklearn.model_selection import train_test_split


X = df.drop('label', axis=1)
y = df['label']
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2,
                                                      stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5,
                                                  stratify=y_temp, random_state=42)

# 10. SAVE PREPROCESSED DATA
df.to_csv('isot_preprocessed.csv', index=False)

In [16]:
df.to_csv('/content/drive/MyDrive/isot_preprocessed.csv', index=False)